In [1]:
import numpy as np
from scipy.sparse import coo_matrix
rng = np.random.default_rng()
print(f"installed solvers:")
%run degdist_to_integer.ipynb import degdist_to_integer

installed solvers:
['CLARABEL', 'HIGHS', 'OSQP', 'SCIP', 'SCIPY', 'SCS']


In [5]:
def edge_to_node(poly):
    """
    converts degree distribution from edge perspective as in density evolution for example
    to distribution from node perspective, 
    """
    int_pol = np.polyint(poly)
    int_pol = int_pol / np.polyval(int_pol, 1)

    idx = np.where(int_pol > 0)[0]
    a = int_pol[idx]
    dv = len(int_pol) - idx - 1

    a = np.flip(a)
    dv = np.flip(dv)

    return dv, a


def getIrregularH(N, lambdA, rho):
    """
    lambda is the degree distribution polynomial from edge perspective and
    the first element is the coefficient corresponding to the largest degree,
    i.e., lambda = [lambda_N, lambda_{N-1}, ..., lambda_2, lambda_1]

    For example, consider
    lambda = [0.5 0 0 0.2 0.3 0]
    This would correspond to a polynomial
                    5        2        
    Lambda(Z) = 0.5 Z  + 0.2 Z  + 0.3 Z

    rho is similarly defined

    For example, to generate the second irregular code example of the lecture
    with n = 10000, we can call
    H = getIrregularH(10000, [0.1151 0.1971 0 0 0.0768 0.202 0.409 0], [1 0 0 0 0 0])

    we can verify that d_{c,avg} = 6 using dcavg = mean(sum(H,2)) and that
    d_{v,avg} = 3 using dvacg = mean(sum(H,1))
    """
    t_dv, t_a_dv = edge_to_node(lambdA)
    t_dc, t_a_dc = edge_to_node(rho)

    dv, n_dv, dc, n_dc = degdist_to_integer(t_dv, t_a_dv, t_dc, t_a_dc, N)

    n_dv = np.round(n_dv).astype(int)
    n_dc = np.round(n_dc).astype(int)

    #E = dv.T @ n_dv

    idx_i = []
    offset = 0
    for k in range(len(dv)):
        temp = np.tile(np.arange(n_dv[k]), dv[k])
        idx_i.append(temp + offset)
        offset += n_dv[k]
    idx_i = np.concatenate(idx_i)

    idx_j = []
    offset = 0
    for k in range(len(dc)):
        temp = np.tile(np.arange(n_dc[k]), dc[k])
        idx_j.append(temp + offset)
        offset += n_dc[k]
    idx_j = np.concatenate(idx_j)

    # find permutation
    permutation = rng.permutation(np.arange(len(idx_j)))

    idx_j = idx_j[permutation]

    abort = False
    while not abort:
        H = coo_matrix((np.ones_like(idx_i), (idx_j, idx_i))).toarray()

        if not np.any(H == 2):
            abort = True
        else:
            # eliminate double edges
            for k in range(N):
                ti = np.where(idx_i == k)[0]

                unique_idx, counts = np.unique(idx_j[ti], return_counts=True)
                duplicates = unique_idx[counts > 1] # duplicates
                if duplicates.size > 0:
                    largeval = duplicates[0]
                    largeidx = np.where(idx_j[ti] == largeval)[0][0]
                    temp = idx_j[ti[largeidx]]
                    rv = rng.integers(idx_j.size)
                    idx_j[ti[largeidx]] = idx_j[rv]
                    idx_j[rv] = temp

    return H

In [7]:
#H = getIrregularH(10000, [0.1151, 0.1971, 0, 0, 0.0768, 0.202, 0.409, 0], [1, 0, 0, 0, 0, 0])
#print(H)